In [2]:
!pip install duckdb pyarrow

Defaulting to user installation because normal site-packages is not writeable


In [6]:
import pandas as pd
import duckdb

print("DuckDB installed successfully!")
print("DuckDB version:", duckdb.__version__)

DuckDB installed successfully!
DuckDB version: 1.5.5


In [7]:
# ============================================================
# TASK 1: CREATE PARQUET FILE
# ============================================================

data = {
    "employee_id": [1, 2, 3, 4, 5, 6, 7, 8],

    "name": [
        "Asha",
        "Rahul",
        "Neha",
        "Vikram",
        "Priya",
        "Arjun",
        "Meera",
        "Karan"
    ],
 "department": [
        "IT",
        "HR",
        "IT",
        "Finance",
        "HR",
        "Finance",
        "IT",
        "Sales"
    ],

    "salary": [
        60000,
        45000,
        70000,
        55000,
        48000,
        65000,
        75000,
        50000
    ],

    "city": [
        "Delhi",
        "Mumbai",
        "Bengaluru",
        "Delhi",
        "Mumbai",
        "Chennai",
        "Bengaluru",
        "Delhi"
    ]
}

df = pd.DataFrame(data)

df.to_parquet(
    "employees.parquet",
    index=False
)

   
print("TASK 1")
print("Parquet file created successfully.")
print(df)


TASK 1
Parquet file created successfully.
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            2   Rahul         HR   45000     Mumbai
2            3    Neha         IT   70000  Bengaluru
3            4  Vikram    Finance   55000      Delhi
4            5   Priya         HR   48000     Mumbai
5            6   Arjun    Finance   65000    Chennai
6            7   Meera         IT   75000  Bengaluru
7            8   Karan      Sales   50000      Delhi


In [8]:
# TASK 2: READ PARQUET USING DUCKDB
print("\nTASK 2")
result = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
""").df()

print(result)


TASK 2
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            2   Rahul         HR   45000     Mumbai
2            3    Neha         IT   70000  Bengaluru
3            4  Vikram    Finance   55000      Delhi
4            5   Priya         HR   48000     Mumbai
5            6   Arjun    Finance   65000    Chennai
6            7   Meera         IT   75000  Bengaluru
7            8   Karan      Sales   50000      Delhi


In [9]:
# TASK 3: FILTER EMPLOYEE RECORDS

print("\nTASK 3.1 - Salary greater than 50000")

high_salary = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    WHERE salary > 50000
""").df()

print(high_salary)


print("\nTASK 3.2 - IT Department")
it_employees = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    WHERE department = 'IT'
""").df()

print(it_employees)


print("\nTASK 3.3 - Employees from Delhi")

delhi_employees = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    WHERE city = 'Delhi'
""").df()

print(delhi_employees)
print("\nTASK 3.4 - IT employees with salary greater than 65000")

it_high_salary = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    WHERE department = 'IT'
      AND salary > 65000
""").df()

print(it_high_salary)


TASK 3.1 - Salary greater than 50000
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            3    Neha         IT   70000  Bengaluru
2            4  Vikram    Finance   55000      Delhi
3            6   Arjun    Finance   65000    Chennai
4            7   Meera         IT   75000  Bengaluru

TASK 3.2 - IT Department
   employee_id   name department  salary       city
0            1   Asha         IT   60000      Delhi
1            3   Neha         IT   70000  Bengaluru
2            7  Meera         IT   75000  Bengaluru

TASK 3.3 - Employees from Delhi
   employee_id    name department  salary   city
0            1    Asha         IT   60000  Delhi
1            4  Vikram    Finance   55000  Delhi
2            8   Karan      Sales   50000  Delhi

TASK 3.4 - IT employees with salary greater than 65000
   employee_id   name department  salary       city
0            3   Neha         IT   70000  Bengaluru
1            7  Meera

In [10]:
# TASK 4: SELECT SPECIFIC COLUMNS
print("\nTASK 4")

selected_columns = duckdb.sql("""
    SELECT name, department, salary
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
""").df()

print(selected_columns)


TASK 4
     name department  salary
0   Meera         IT   75000
1    Neha         IT   70000
2   Arjun    Finance   65000
3    Asha         IT   60000
4  Vikram    Finance   55000
5   Karan      Sales   50000
6   Priya         HR   48000
7   Rahul         HR   45000


In [11]:
# TASK 5: AGGREGATIONS
print("\nTASK 5")
summary = duckdb.sql("""
    SELECT
        COUNT(*) AS employee_count,
        AVG(salary) AS average_salary,
        MAX(salary) AS maximum_salary,
        MIN(salary) AS minimum_salary,
        SUM(salary) AS total_salary
    FROM read_parquet('employees.parquet')
""").df()
print(summary)


TASK 5
   employee_count  average_salary  maximum_salary  minimum_salary  \
0               8         58500.0           75000           45000   

   total_salary  
0      468000.0  


In [12]:
# TASK 6: GROUP DATA BY DEPARTMENT
print("\nTASK 6")

department_summary = duckdb.sql("""
    SELECT
        department,
        COUNT(*) AS employee_count,
        AVG(salary) AS average_salary,
        MAX(salary) AS highest_salary,
        SUM(salary) AS total_salary
    FROM read_parquet('employees.parquet')
    GROUP BY department
    ORDER BY average_salary DESC
""").df()

print(department_summary)



TASK 6
  department  employee_count  average_salary  highest_salary  total_salary
0         IT               3    68333.333333           75000      205000.0
1    Finance               2    60000.000000           65000      120000.0
2      Sales               1    50000.000000           50000       50000.0
3         HR               2    46500.000000           48000       93000.0


In [13]:
 # TASK 7: CREATE DUCKDB DATABASE

print("\nTASK 7")

connection = duckdb.connect("company.duckdb")

connection.execute("""
    CREATE OR REPLACE TABLE employees AS
    SELECT *
    FROM read_parquet('employees.parquet')
""")

result = connection.execute("""
    SELECT *
    FROM employees
""").df()

print(result)

connection.close()

print("company.duckdb created successfully.")


TASK 7
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            2   Rahul         HR   45000     Mumbai
2            3    Neha         IT   70000  Bengaluru
3            4  Vikram    Finance   55000      Delhi
4            5   Priya         HR   48000     Mumbai
5            6   Arjun    Finance   65000    Chennai
6            7   Meera         IT   75000  Bengaluru
7            8   Karan      Sales   50000      Delhi
company.duckdb created successfully.


In [14]:
# TASK 8: EXPORT HIGH-SALARY EMPLOYEES
print("\nTASK 8")

duckdb.sql("""
    COPY (
        SELECT *
        FROM read_parquet('employees.parquet')
        WHERE salary > 50000
    )
    TO 'high_salary_employees.parquet'
    (FORMAT PARQUET)
""")

print("high_salary_employees.parquet created successfully.")



TASK 8
high_salary_employees.parquet created successfully.


In [21]:
# TASK 9: VERIFY EXPORTED FILE\
print("\nTASK 9")

verified_result = duckdb.sql("""
    SELECT *
    FROM read_parquet('high_salary_employees.parquet')
""").df()

print(verified_result)
print("high_salary_employees.parquet verified successfully.")


TASK 9
   employee_id    name department  salary       city
0            1    Asha         IT   60000      Delhi
1            3    Neha         IT   70000  Bengaluru
2            4  Vikram    Finance   55000      Delhi
3            6   Arjun    Finance   65000    Chennai
4            7   Meera         IT   75000  Bengaluru
high_salary_employees.parquet verified successfully.


In [16]:
# BONUS 1: SECOND-HIGHEST SALARY
print("\nBONUS 1")

second_highest = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
    LIMIT 1 OFFSET 1
""").df()

print(second_highest)


BONUS 1
   employee_id  name department  salary       city
0            3  Neha         IT   70000  Bengaluru


In [17]:
# BONUS 2: TOP 3 HIGHEST-PAID EMPLOYEES
print("\nBONUS 2")

top_three = duckdb.sql("""
    SELECT *
    FROM read_parquet('employees.parquet')
    ORDER BY salary DESC
    LIMIT 3
""").df()

print(top_three)


BONUS 2
   employee_id   name department  salary       city
0            7  Meera         IT   75000  Bengaluru
1            3   Neha         IT   70000  Bengaluru
2            6  Arjun    Finance   65000    Chennai


In [18]:
# BONUS 3: AVERAGE SALARY FOR EACH 
print("\nBONUS 3")

city_salary = duckdb.sql("""
    SELECT
        city,
        AVG(salary) AS average_salary
    FROM read_parquet('employees.parquet')
    GROUP BY city
    ORDER BY average_salary DESC
""").df()

print(city_salary)



BONUS 3
        city  average_salary
0  Bengaluru         72500.0
1    Chennai         65000.0
2      Delhi         55000.0
3     Mumbai         46500.0


In [19]:
# BONUS 4: DEPARTMENTS WITH AVERAGE SALARY > 55000
print("\nBONUS 4")

high_average_departments = duckdb.sql("""
    SELECT
        department,
        AVG(salary) AS average_salary
    FROM read_parquet('employees.parquet')
    GROUP BY department
    HAVING AVG(salary) > 55000
""").df()

print(high_average_departments)


BONUS 4
  department  average_salary
0         IT    68333.333333
1    Finance    60000.000000


In [20]:
# BONUS 5: SALARY CATEGORY
print("\nBONUS 5")

salary_category = duckdb.sql("""
    SELECT
        name,
        salary,
        CASE
            WHEN salary >= 65000 THEN 'High'
            WHEN salary >= 50000 THEN 'Medium'
            ELSE 'Low'
        END AS salary_category
    FROM read_parquet('employees.parquet')
""").df()

print(salary_category)
print("TASK 11 COMPLETED SUCCESSFULLY")


BONUS 5
     name  salary salary_category
0    Asha   60000          Medium
1   Rahul   45000             Low
2    Neha   70000            High
3  Vikram   55000          Medium
4   Priya   48000             Low
5   Arjun   65000            High
6   Meera   75000            High
7   Karan   50000          Medium
TASK 11 COMPLETED SUCCESSFULLY
